# Jev versus eleven conventional classification pipelines — v2

Use a **fresh Kaggle session**, Internet enabled, **T4 × 2**, and an enabled secret named `TYPESAFE_API_KEY`.

This notebook extracts its bundled Python modules; no separate code upload is required. The full editable bundle is also available. All ML tuning uses development data. One fixed test set is shared across models and training seeds. The previous run should be retained as a pilot.

**Run the ML cells first.** The separate Jev cell makes paid API requests. Full mode plans about 52,410 requests before cache reuse/retries on the current snapshots. Check the displayed counts and your pricing. The estimate guard is not a billing cap.

This is a modest-budget comparison, not a claim of best achievable ML performance.

In [ ]:
# Extract the frozen source package. Run before importing numerical libraries.
import base64, hashlib, io, sys, zipfile
from pathlib import Path

# Optional: point at an edited bundle uploaded as a Kaggle Dataset.
# The directory must contain jevbench/, requirements-v2.txt, and BENCHMARK_V2.md.
EXTERNAL_CODE_DIR = None
PACKAGE_SHA256 = 'ca8a01dea7550610e814e6f3c025e0f1fc96ce111e6ce91cf4062c765163a53f'
if EXTERNAL_CODE_DIR:
    CODE_DIR = Path(EXTERNAL_CODE_DIR)
else:
    bundled = base64.b64decode('...')
    assert hashlib.sha256(bundled).hexdigest() == PACKAGE_SHA256
    CODE_DIR = Path('/kaggle/working') / ('jevbench_code_' + PACKAGE_SHA256[:12])
    CODE_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(bundled)) as archive:
        archive.extractall(CODE_DIR)
assert (CODE_DIR / 'jevbench' / 'runner.py').exists()
sys.path.insert(0, str(CODE_DIR))
print('Source modules:', CODE_DIR)

In [ ]:
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE_DIR / 'requirements-v2.txt')])
# If you imported numpy/sklearn before installation, restart the session and rerun.


## Freeze the run configuration

`benchmark` uses three seeds, four candidates, up to 400 trees, 12,000 training rows, 1,500 model-selection rows, and 500 decision-policy rows. `quick` is for smoke testing only. Test caps are 1,000 per dataset and 1,500 for Banking77; small datasets retain fewer rows.

Default pilot exclusion assumes seeds 42/43/44, test cap 300 and matching snapshots. If your pilot differed, use its actual saved IDs in `datasets.py` before running. Changing the configuration, source or environment requires a new `ROOT`.

In [ ]:
from IPython.display import display, Markdown, FileLink
from jevbench.config import configuration
from jevbench.runner import prepare_suite, run_ml, run_jev
from jevbench.reporting import render_results

CFG = configuration('benchmark')
ROOT = Path('/kaggle/working/jev_benchmark_v2')
# Optional existing pilot output directory: reuses data snapshots, not pilot scores.
PILOT_ROOT = None
# Example: PILOT_ROOT = Path('/kaggle/input/my-pilot-results/jev_benchmark_parallel')

# Two T4s: independent worker per GPU; CPU models use two threads per worker.
CFG['max_parallel_jobs'] = 2
CFG['threads'] = 2
CFG['jev_model'] = 'jev-1.13.0'
display(CFG)
display(Markdown((CODE_DIR / 'BENCHMARK_V2.md').read_text()))

In [ ]:
OVERVIEW = prepare_suite(ROOT, CFG, pilot_root=PILOT_ROOT)
display(OVERVIEW)


## Fit and tune conventional models

GPU probes precede training. Successful per-model checkpoints allow rerunning this cell after interruption. Individual trial warnings/errors are saved under `runs/`; inspect them before publishing. Most scikit-learn models use CPUs. Larger text forests can take hours.

In [ ]:
ML_STATUS = run_ml(ROOT, CFG)
display(ML_STATUS)
TABLES = render_results(ROOT, CFG)
display(TABLES['raw_balanced_accuracy'])
display(TABLES['adjusted_balanced_accuracy'])

## Run Jev — paid API calls

The secret is read from Kaggle Secrets. tqdm shows each test/policy partition; exact successful requests are cached. Permanent API errors stop execution; exhausted transient retries count as failed predictions. Model IDs, request attempts and cache hits are recorded.

Binary **adjusted Jev uses labeled policy data to select a threshold**. Only the raw zero-shot panel is an end-to-end zero-shot baseline. Few-shot examples are sampled from training, one per class. Budget estimates exclude output-token charges; verify current pricing and account limits before executing this cell.

In [ ]:
API_STATUS = run_jev(ROOT, CFG)
display(API_STATUS)

## Compare and export

Rows are datasets; columns are models. Values are mean ± sample SD across training seeds, **not confidence intervals**. Both panels use the same test cases. The report includes accuracy/macro-F1 and paired bootstrap differences; JSON records include minority recall, confusion matrices, thresholds and raw probabilities. Publish both raw and adjusted panels with the protocol.

Save/download the output ZIP to preserve caches and resume later. It contains benchmark artifacts and dataset snapshots, never the API key.

In [ ]:
TABLES = render_results(ROOT, CFG)
display(Markdown('### Raw decision rules'))
display(TABLES['raw_balanced_accuracy'])
display(Markdown('### Binary thresholds selected on policy data'))
display(TABLES['adjusted_balanced_accuracy'])
display(FileLink(str(ROOT / 'report.html')))
import shutil
# Include the source/protocol used in the result download for reproducibility.
shutil.copytree(CODE_DIR / 'jevbench', ROOT / 'source' / 'jevbench', dirs_exist_ok=True,
                ignore=shutil.ignore_patterns('__pycache__'))
for filename in ['BENCHMARK_V2.md', 'requirements-v2.txt']:
    shutil.copy2(CODE_DIR / filename, ROOT / filename)
archive = shutil.make_archive(str(ROOT) + '_results', 'zip', root_dir=ROOT.parent, base_dir=ROOT.name)
display(FileLink(archive))